# MLP and Transformer 12-Step Forecasting

This notebook uses one shared BasicTS forecasting data pipeline for both models. The MLP section comes first and is the first model to shape-check and train.


## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [2]:
import os
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
os.chdir(ROOT)

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("CWD:", Path.cwd())


CWD: C:\Users\luwil\OneDrive\Documents\Code\BasicTS


## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, and `output_len`. I keep `use_timestamps=False` so each model receives exactly `[batch_size, input_len, num_features]`.


In [3]:
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode

DATASET_NAME = "ETTh1"
INPUT_LEN = 96
OUTPUT_LEN = 12
NUM_FEATURES = 7
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "output_len": OUTPUT_LEN,
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
}

print({
    "dataset": DATASET_NAME,
    "input_len": INPUT_LEN,
    "output_len": OUTPUT_LEN,
    "num_features": NUM_FEATURES,
    "scaler": "ZScoreScaler",
    "same_preprocessing": "BasicTS forecasting taskflow",
})


{'dataset': 'ETTh1', 'input_len': 96, 'output_len': 12, 'num_features': 7, 'scaler': 'ZScoreScaler', 'same_preprocessing': 'BasicTS forecasting taskflow'}


## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [4]:
def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }


def preview_shapes(cfg):
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    raw_batch = _float_batch(next(iter(train_loader)))

    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)

    class PreviewRunner:
        pass

    runner = PreviewRunner()
    runner.cfg = cfg
    runner.scaler = scaler

    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    model = cfg.model(cfg.model_config)
    model.eval()

    with torch.no_grad():
        prediction = model(processed_batch["inputs"])
        if isinstance(prediction, dict):
            prediction = prediction["prediction"]

    print("raw inputs:        ", tuple(raw_batch["inputs"].shape))
    print("raw targets:       ", tuple(raw_batch["targets"].shape))
    print("processed inputs:  ", tuple(processed_batch["inputs"].shape))
    print("processed targets: ", tuple(processed_batch["targets"].shape))
    print("model prediction:  ", tuple(prediction.shape))

    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


def metrics_file(cfg):
    return Path(cfg.ckpt_save_dir) / cfg.md5 / "test_metrics.json"


def load_test_metrics(cfg):
    path = metrics_file(cfg)
    if not path.exists():
        print(f"No test metrics found yet: {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


## 4. MLP Model

The MLP flattens `[batch_size, input_len, num_features]`, passes the flat vector through linear layers, and reshapes the head output back to `[batch_size, 12, num_features]`.


In [5]:
class SimpleMLPForecaster(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_len = config.input_len
        self.output_len = config.output_len
        self.num_features = config.num_features
        flat_input = self.input_len * self.num_features
        flat_output = self.output_len * self.num_features

        self.net = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(flat_input, config.hidden_size),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, flat_output),
        )

    def forward(self, inputs):
        prediction = self.net(inputs)
        return prediction.view(inputs.size(0), self.output_len, self.num_features)


## 5. MLP Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [6]:
mlp_model_config = BasicTSModelConfig(
    input_len=INPUT_LEN,
    output_len=OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=256,
    dropout=0.1,
)

mlp_cfg = BasicTSForecastingConfig(
    model=SimpleMLPForecaster,
    model_config=mlp_model_config,
    ckpt_save_dir=f"checkpoints/SimpleMLPForecaster/{DATASET_NAME}_{INPUT_LEN}_{OUTPUT_LEN}",
    **SHARED_CONFIG,
)

mlp_cfg


BasicTSForecastingConfig(model=<class '__main__.SimpleMLPForecaster'>, model_config={'input_len': 96, 'output_len': 12, 'num_features': 7, 'hidden_size': 256, 'dropout': 0.1}, dataset_name='ETTh1', taskflow=<basicts.runners.taskflow.forecasting_taskflow.BasicTSForecastingTaskFlow object at 0x000001E0251D08D0>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epochs=5, num_steps=None, loss='MAE', optimizer=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.001, 'weight_decay': 0.0005}, lr=None, lr_scheduler=None, lr_scheduler_pa

## 6. MLP Shape Test

Run this before training. The final printed line must be `(batch_size, 12, num_features)`.


In [7]:
mlp_batch, mlp_prediction = preview_shapes(mlp_cfg)


raw inputs:         (32, 96, 7)
raw targets:        (32, 12, 7)
processed inputs:   (32, 96, 7)
processed targets:  (32, 12, 7)
model prediction:   (32, 12, 7)


## 7. Train the MLP

This is the first training run. Leave `RUN_MLP_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [13]:
RUN_MLP_TRAINING = True

if RUN_MLP_TRAINING:
    BasicTSLauncher.launch_training(mlp_cfg)
else:
    print("MLP training skipped. Set RUN_MLP_TRAINING = True to train.")


2026-06-12 19:05:04,499 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-12 19:05:04,521 - BasicTS - INFO - Building model.
2026-06-12 19:05:04,532 - BasicTS - INFO - Set ckpt save dir: "checkpoints/SimpleMLPForecaster/ETTh1_96_12\438bbc0ba98a18486eea509797edf57c"
2026-06-12 19:05:04,534 - BasicTS-training - INFO - Initializing training.
2026-06-12 19:05:04,548 - BasicTS-training - INFO - Building train data loader.
2026-06-12 19:05:06,149 - BasicTS-training - INFO - Set optim: Adam
2026-06-12 19:05:06,150 - BasicTS-training - INFO - Building val data loader.
2026-06-12 19:05:06,157 - BasicTS-training - INFO - Building test data loader.
2026-06-12 19:05:06,165 - BasicTS-training - INFO - Total parameters: 259668
2026-06-12 19:05:06,166 - BasicTS-training - INFO - Trainable parameters: 259668
2026-06-12 19:05:06,167 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [00:01<00:00, 185.42it/s]
2026-06-12 19:05:07,614 - BasicTS-training - INFO - Result <train>:

## 8. Transformer Model

After the MLP shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 12, num_features]`.


In [14]:
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    ckpt_save_dir=f"checkpoints/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_{OUTPUT_LEN}",
    **SHARED_CONFIG,
)

transformer_cfg


BasicTSForecastingConfig(model=<class 'basicts.models.iTransformer.arch.itransformer_arch.iTransformerForForecasting'>, model_config=iTransformerConfig(input_len=96, output_len=12, num_features=7, num_classes=None, hidden_size=64, n_heads=4, intermediate_size=128, hidden_act='gelu', num_layers=2, dropout=0.1, use_revin=True, output_attentions=False), dataset_name='ETTh1', taskflow=<basicts.runners.taskflow.forecasting_taskflow.BasicTSForecastingTaskFlow object at 0x000001E0251D08D0>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epo

## 9. Transformer Shape Test

Run this after the MLP section works. It uses the same shared data pipeline and checks the same input/output shape contract.


In [10]:
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg)


raw inputs:         (32, 96, 7)
raw targets:        (32, 12, 7)
processed inputs:   (32, 96, 7)
processed targets:  (32, 12, 7)
model prediction:   (32, 12, 7)


## 10. Train the Transformer

Use the same dataset, scaler, preprocessing, `input_len`, and `output_len` as the MLP. Keep this off until the MLP has trained successfully.


In [15]:
RUN_TRANSFORMER_TRAINING = True

if RUN_TRANSFORMER_TRAINING:
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("Transformer training skipped. Set RUN_TRANSFORMER_TRAINING = True after the MLP works.")


2026-06-12 19:05:45,728 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-12 19:05:45,745 - BasicTS - INFO - Building model.
2026-06-12 19:05:45,748 - BasicTS - INFO - Set ckpt save dir: "checkpoints/iTransformerForForecasting/ETTh1_96_12\a617f098e6e99a02765c370f7391f0b9"
2026-06-12 19:05:45,749 - BasicTS-training - INFO - Initializing training.
2026-06-12 19:05:45,750 - BasicTS-training - INFO - Building train data loader.
2026-06-12 19:05:45,755 - BasicTS-training - INFO - Set optim: Adam
2026-06-12 19:05:45,757 - BasicTS-training - INFO - Building val data loader.
2026-06-12 19:05:45,758 - BasicTS-training - INFO - Building test data loader.
2026-06-12 19:05:45,761 - BasicTS-training - INFO - Total parameters: 74060
2026-06-12 19:05:45,762 - BasicTS-training - INFO - Trainable parameters: 74060
2026-06-12 19:05:45,763 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [00:04<00:00, 62.77it/s]
2026-06-12 19:05:50,019 - BasicTS-training - INFO - Result <tra

## 11. Hybrid Prediction: MLP First 6, Transformer Next 6

This cell combines the saved test predictions from both trained models. The first 6 forecast steps come from the MLP, and the next 6 forecast steps come from the Transformer, so the final hybrid prediction still has shape `[num_test_samples, 12, num_features]`.


In [ ]:
import numpy as np


def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]


def load_basicts_prediction(cfg):
    prediction_path = latest_prediction_file(cfg)
    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    prediction_shape = (len(test_dataset), cfg.output_len, NUM_FEATURES)

    # BasicTS writes this file as a raw memmap, even though the file name ends in .npy.
    prediction = np.memmap(
        prediction_path,
        dtype=np.float32,
        mode="r",
        shape=prediction_shape,
    )
    return prediction_path, np.asarray(prediction)


mlp_pred_path, mlp_pred = load_basicts_prediction(mlp_cfg)
transformer_pred_path, transformer_pred = load_basicts_prediction(transformer_cfg)

print("MLP prediction file:", mlp_pred_path)
print("Transformer prediction file:", transformer_pred_path)
print("MLP prediction shape:", mlp_pred.shape)
print("Transformer prediction shape:", transformer_pred.shape)

assert mlp_pred.shape == transformer_pred.shape
assert mlp_pred.shape[1] == OUTPUT_LEN
assert OUTPUT_LEN == 12

hybrid_pred = np.concatenate(
    [
        mlp_pred[:, :6, :],
        transformer_pred[:, 6:12, :],
    ],
    axis=1,
)

print("Hybrid prediction shape:", hybrid_pred.shape)

hybrid_save_path = Path("checkpoints/hybrid_mlp_first6_transformer_next6_ETTh1_96_12_prediction.npy")
np.save(hybrid_save_path, hybrid_pred)
print("Saved hybrid prediction to:", hybrid_save_path)


## 12. MAE/MSE Comparison

After the hybrid prediction is created, this cell compares the saved test metrics for the two trained models and computes MAE/MSE for the hybrid prediction against the same saved test targets.


In [ ]:
def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        print(f"No test metrics found under {cfg.ckpt_save_dir}")
        return None
    return metrics_files[0]


def load_latest_test_metrics(cfg):
    path = latest_metrics_file(cfg)
    if path is None:
        return None
    with path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


def load_basicts_targets(prediction_path, shape):
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")
    targets = np.memmap(targets_path, dtype=np.float32, mode="r", shape=shape)
    return targets_path, np.asarray(targets)


targets_path, hybrid_targets = load_basicts_targets(mlp_pred_path, hybrid_pred.shape)
hybrid_metrics = {
    "MAE": float(np.mean(np.abs(hybrid_pred - hybrid_targets))),
    "MSE": float(np.mean((hybrid_pred - hybrid_targets) ** 2)),
}

comparison = {
    "MLP": load_latest_test_metrics(mlp_cfg),
    "Transformer": load_latest_test_metrics(transformer_cfg),
    "Hybrid (MLP first 6, Transformer last 6)": hybrid_metrics,
}

print("Targets file:", targets_path)

for model_name, metrics in comparison.items():
    if metrics is None:
        continue
    print(model_name)
    for metric_name in ["MAE", "MSE"]:
        if metric_name in metrics:
            print(f"  {metric_name}: {metrics[metric_name]:.6f}")


## 13. Efficiency Comparison

This cell compares model size and average batch prediction time, then lines those numbers up with the MAE/MSE results. The hybrid uses both saved model predictions, so its inference time is estimated as MLP time plus Transformer time.


In [ ]:
import time


def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def time_model_prediction(model, batch, repeats=50, warmup=5):
    device = next(model.parameters()).device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            _ = model(inputs)
    end = time.perf_counter()

    return (end - start) / repeats


mlp_efficiency_model = mlp_cfg.model(mlp_cfg.model_config)
transformer_efficiency_model = transformer_cfg.model(transformer_cfg.model_config)

mlp_params = count_trainable_parameters(mlp_efficiency_model)
transformer_params = count_trainable_parameters(transformer_efficiency_model)

mlp_time = time_model_prediction(mlp_efficiency_model, mlp_batch)
transformer_time = time_model_prediction(transformer_efficiency_model, transformer_batch)
hybrid_time = mlp_time + transformer_time

mlp_metrics = comparison.get("MLP", {}) or {}
transformer_metrics = comparison.get("Transformer", {}) or {}
hybrid_metrics = comparison.get("Hybrid (MLP first 6, Transformer last 6)", {}) or {}

rows = [
    ("MLP", mlp_metrics, mlp_params, mlp_time),
    ("Transformer", transformer_metrics, transformer_params, transformer_time),
    ("Hybrid", hybrid_metrics, mlp_params + transformer_params, hybrid_time),
]

print(f"{'Model':<14} {'MAE':>10} {'MSE':>10} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 65)
for model_name, metrics, params, avg_time in rows:
    mae = metrics.get("MAE", float("nan"))
    mse = metrics.get("MSE", float("nan"))
    print(f"{model_name:<14} {mae:>10.6f} {mse:>10.6f} {params:>12,} {avg_time:>15.6f}")
